# Stage 00b1 — Grouping, Structured Symlinks & Batch Assignment

What this notebook does:
1. Runs grouper.run_grouping() to assign company_canonical and prototype_label to every patent
2. Creates symlinks: structured/{company_canonical}/{prototype_label}/{patent_id}/ → matched/{patent_id}/
3. Assigns batch_id so that patents in the same cluster stay together, with ≥300 patents per batch
4. Writes data/batches.xlsx — one sheet per batch, plus a Summary sheet

Run after: 00b (matched/ must exist)
Run before: 01_review

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}.")

for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))
# deduplicator.py is currently archived (not part of the active src/ package).
# Add src/_archive to sys.path so cell 2's `from deduplicator import run_deduplication` resolves.
sys.path.insert(0, str(repo_root / "src" / "_archive"))

_cl_spec = importlib.util.spec_from_file_location("config_loader", repo_root / "src" / "config_loader.py")
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

cfg = load_config()
print(f"matched    : {cfg['paths']['matched']}")
print(f"structured : {cfg['paths']['structured']}")
print(f"reviewed   : {cfg['paths']['reviewed']}")

In [ ]:
# ── Load Excel + run grouper ───────────────────────────────────────────────────
import pandas as pd
from extractor import load_patseer_excel
from deduplicator import run_deduplication
from grouper import run_grouping

raw_df = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
deduped_df, _ = run_deduplication(raw_df, cfg)
grouped_df    = run_grouping(deduped_df, cfg)

print(f"\nColumns available: {list(grouped_df.columns)}")
print(f"Total patents after dedup: {len(grouped_df)}")
print(f"\nTop clusters:")
print(grouped_df.groupby(["company_canonical","prototype_label"]).size().sort_values(ascending=False).head(20).to_string())

In [ ]:
# ── Assign batch_id (≥300 per batch, clusters kept together) ──────────────────
# Strategy: sort by display_order (clusters already grouped), walk forward
# and cut a new batch whenever we've collected ≥300 AND we just finished a cluster.

MIN_BATCH_SIZE = 300

grouped_df = grouped_df.sort_values("display_order").reset_index(drop=True)

batch_ids   = []
batch_id    = 1
count       = 0
prev_cluster = None

for _, row in grouped_df.iterrows():
    cluster = (row["company_canonical"], row["prototype_label"])
    # Cut batch when we hit a new cluster AND have enough patents already
    if cluster != prev_cluster and count >= MIN_BATCH_SIZE:
        batch_id += 1
        count = 0
    batch_ids.append(batch_id)
    count += 1
    prev_cluster = cluster

grouped_df["batch_id"] = batch_ids

print(f"Total batches: {grouped_df['batch_id'].max()}")
print(grouped_df.groupby("batch_id").size().rename("patent_count").to_string())

In [ ]:
# ── Create structured/ symlinks ────────────────────────────────────────────────
import os

matched_root    = Path(cfg["paths"]["matched"])
structured_root = Path(cfg["paths"]["structured"])

_PUB_VARIANTS = ["Publication Number", "Pub. No.", "Patent Number", "Record Number"]
pub_col = next((c for c in _PUB_VARIANTS if c in grouped_df.columns), None)
if pub_col is None:
    raise KeyError(f"No pub-number column found. Columns: {list(grouped_df.columns)}")

created  = 0
skipped  = 0
missing  = 0

for _, row in grouped_df.iterrows():
    patent_id = str(row[pub_col]).strip()
    company   = str(row["company_canonical"]).strip()
    prototype = str(row["prototype_label"]).strip()

    src_dir  = matched_root / patent_id
    link_dir = structured_root / company / prototype / patent_id

    if not src_dir.exists():
        missing += 1
        continue

    link_dir.parent.mkdir(parents=True, exist_ok=True)

    if link_dir.exists() or link_dir.is_symlink():
        skipped += 1
        continue

    os.symlink(src_dir.resolve(), link_dir)
    created += 1

print(f"Symlinks created : {created}")
print(f"Already existed  : {skipped}")
print(f"matched/ missing : {missing}  (run 00b first for these)")

In [ ]:
# ── Write data/batches.xlsx ───────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font
from openpyxl.utils.dataframe import dataframe_to_rows

excel_index = load_patseer_excel(cfg["paths"]["patseer_excel"])
data_dir    = Path(cfg["paths"]["data"])
data_dir.mkdir(parents=True, exist_ok=True)
out_path    = data_dir / "batches.xlsx"

wb = Workbook()
# Remove default sheet
wb.remove(wb.active)

# ── Summary sheet ─────────────────────────────────────────────────────────────
summary_rows = []
for bid, grp in grouped_df.groupby("batch_id"):
    summary_rows.append({
        "batch_id":      int(bid),
        "patent_count":  len(grp),
        "companies":     ", ".join(sorted(grp["company_canonical"].unique())),
        "clusters":      ", ".join(sorted(grp["prototype_label"].unique())),
        "review_status": "pending",
    })

ws_sum = wb.create_sheet("Summary")
hdr_fill = PatternFill("solid", fgColor="1F497D")
hdr_font = Font(color="FFFFFF", bold=True)
sum_cols  = list(summary_rows[0].keys())
ws_sum.append(sum_cols)
for cell in ws_sum[1]:
    cell.fill = hdr_fill; cell.font = hdr_font
for r in summary_rows:
    ws_sum.append([r[c] for c in sum_cols])

# ── Per-batch sheets ──────────────────────────────────────────────────────────
BATCH_COLS = [
    "patent_id", "company_canonical", "prototype_label", "batch_id",
    "image_files", "description_of_drawings",
    "ai_label_json", "human_review_status", "notes",
]

for bid, grp in grouped_df.groupby("batch_id"):
    ws = wb.create_sheet(f"Batch_{int(bid):02d}")
    ws.append(BATCH_COLS)
    for cell in ws[1]:
        cell.fill = hdr_fill; cell.font = hdr_font

    for _, row in grp.iterrows():
        pid     = str(row[pub_col]).strip()
        meta    = excel_index.get(pid, {})
        img_dir = matched_root / pid
        imgs    = sorted(img_dir.glob("*.png")) if img_dir.exists() else []

        ws.append([
            pid,
            str(row["company_canonical"]),
            str(row["prototype_label"]),
            int(bid),
            ", ".join(f.name for f in imgs),
            meta.get("description_of_drawings", ""),
            "",   # ai_label_json — filled by 01_review
            "pending",
            "",
        ])

wb.save(out_path)
print(f"Saved {out_path}")
print(f"Sheets: {wb.sheetnames}")